# 持仓监控实战

本案例展示如何使用持仓监控系统实时监控持仓状态。

## 核心功能
- 添加/删除持仓
- 实时监控盈亏
- 自动止损止盈预警
- 资金流向监控

## 1. 环境准备

In [ ]:
import os
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'scripts'))

# 检查环境变量
print("=== 环境变量检查 ===")
print(f"妙想API Key: {'已配置' if os.environ.get('MX_APIKEY') else '未配置'}")
print(f"问财API Key: {'已配置' if os.environ.get('IWENCAI_API_KEY') else '未配置'}")

print("\n✅ 环境准备完成")

## 2. 初始化持仓监控器

In [ ]:
from position_monitor import PositionMonitor

# 创建监控器实例
monitor = PositionMonitor()
print("✅ 持仓监控器初始化成功")

## 3. 添加持仓

In [ ]:
# 添加持仓示例
positions = [
    {
        "stock_code": "600519",
        "stock_name": "贵州茅台",
        "shares": 100,
        "cost": 1800.0,  # 成本价
    },
    {
        "stock_code": "300750",
        "stock_name": "宁德时代",
        "shares": 200,
        "cost": 200.0,
    },
]

for pos in positions:
    monitor.add_position(
        stock_code=pos["stock_code"],
        stock_name=pos["stock_name"],
        shares=pos["shares"],
        cost=pos["cost"]
    )
    print(f"✅ 已添加: {pos['stock_name']}({pos['stock_code']})")

print(f"\n当前持仓数: {len(monitor.positions)}")

## 4. 查看持仓状态

In [ ]:
# 获取持仓状态
status = monitor.get_positions_status()

print("\n=== 持仓状态 ===")
print(f"{'股票':<12} {'持仓':<8} {'成本':<10} {'现价':<10} {'盈亏':<10} {'盈亏率':<10}")
print("-" * 70)

for pos in status:
    print(f"{pos['stock_name']:<12} {pos['shares']:<8} {pos['cost']:<10.2f} "
          f"{pos['current_price']:<10.2f} {pos['profit_loss']:<10.2f} {pos['profit_loss_pct']:<10.2%}")

# 总盈亏
total_profit = sum(p['profit_loss'] for p in status)
print(f"\n总盈亏: {total_profit:.2f}元")

## 5. 设置止损止盈

In [ ]:
# 设置止损止盈规则
rules = {
    "stop_loss_pct": -0.05,  # 止损: -5%
    "take_profit_pct": 0.10,  # 止盈: +10%
    "trailing_stop_pct": 0.03,  # 移动止损: 3%
}

monitor.set_alert_rules(rules)
print("✅ 止损止盈规则已设置")
print(f"  止损线: {rules['stop_loss_pct']:.2%}")
print(f"  止盈线: {rules['take_profit_pct']:.2%}")
print(f"  移动止损: {rules['trailing_stop_pct']:.2%}")

## 6. 检查预警

In [ ]:
# 检查预警
alerts = monitor.check_alerts()

if alerts:
    print("\n=== 预警信息 ===")
    for alert in alerts:
        severity_emoji = "🔴" if alert['severity'] == "HIGH" else "🟡"
        print(f"{severity_emoji} {alert['stock_name']}({alert['stock_code']})")
        print(f"   类型: {alert['alert_type']}")
        print(f"   消息: {alert['message']}")
        print(f"   时间: {alert['timestamp']}")
        print()
else:
    print("\n✅ 无预警")

## 7. 资金流向监控

In [ ]:
# 检查持仓股票的资金流向
fund_flows = monitor.check_fund_flows()

print("\n=== 资金流向监控 ===")
print(f"{'股票':<12} {'主力流入':<12} {'DDX':<10} {'趋势':<10}")
print("-" * 50)

for flow in fund_flows:
        trend = "流入 ✅" if flow['main_inflow'] > 0 else "流出 ❌"
    print(f"{flow['stock_name']:<12} {flow['main_inflow']:>10.2f}亿 {flow['ddx']:>10.2f} {trend:<10}")

## 8. 生成监控报告

In [ ]:
# 生成报告
report = monitor.generate_report()

print("\n=== 持仓监控报告 ===")
print(f"生成时间: {report['timestamp']}")
print(f"\n持仓概况:")
print(f"  总市值: {report['total_market_value']:.2f}元")
print(f"  总盈亏: {report['total_profit_loss']:.2f}元")
print(f"  盈亏率: {report['total_profit_loss_pct']:.2%}")
print(f"\n预警统计:")
print(f"  高风险: {report['alert_stats']['HIGH']}只")
print(f"  中风险: {report['alert_stats']['MEDIUM']}只")
print(f"  低风险: {report['alert_stats']['LOW']}只")
print(f"\n资金流向:")
print(f"  流入股票: {report['fund_flow_stats']['inflow']}只")
print(f"  流出股票: {report['fund_flow_stats']['outflow']}只")

## 9. 持续监控（可选）

In [ ]:
# 持续监控（每5分钟检查一次）
import time

duration = input("监控时长（分钟，默认5分钟）: ")
duration = int(duration) if duration else 5

print(f"\n开始监控 {duration} 分钟...")

for i in range(duration):
    print(f"\n[{i+1}/{duration}] 检查中...")
    
    # 检查预警
    alerts = monitor.check_alerts()
    if alerts:
        for alert in alerts:
            print(f"  ⚠️ {alert['stock_name']}: {alert['message']}")
    
    # 检查资金流向
    flows = monitor.check_fund_flows()
    for flow in flows:
        if flow['main_inflow'] < -100000000:  # 流出超1亿
            print(f"  ❌ {flow['stock_name']}: 主力流出{abs(flow['main_inflow'])/1e8:.2f}亿")
    
    if i < duration - 1:
        time.sleep(60)  # 等待1分钟

print("\n✅ 监控结束")

## 总结

本案例展示了持仓监控的完整流程：
1. 添加持仓
2. 查看持仓状态
3. 设置止损止盈
4. 检查预警
5. 资金流向监控
6. 生成监控报告
7. 持续监控

### 止损止盈规则
- 止损: -5%（亏损超过5%触发预警）
- 止盈: +10%（盈利超过10%提醒减仓）
- 移动止损: 3%（利润回撤3%提醒卖出）

### 预警类型
- STOP_LOSS: 触发止损线
- TAKE_PROFIT: 触发止盈线
- PRICE_CHANGE: 价格大幅波动
- FUND_FLOW: 资金流向异常